# Подготовка данных RFSD для BI-дашборда

**Источник:** Russian Financial Statements Database (RFSD), версия 3.1.0, Hugging Face `irlspbru/RFSD`.
**Период:** 2015–2025 (10 лет).
**Цель:** очистить сырые parquet-файлы, привести показатели к единому виду и собрать модель «звезда» (факт + измерения) для загрузки в Yandex DataLens.

**Что делаем:**
1. Загружаем parquet по годам.
2. Осматриваем данные: колонки, типы, пропуски.
3. Переименовываем технические `line_*` в понятные имена и переводим суммы из тысяч рублей в рубли.
4. Финально проверяем данные на дубликаты.
5. Строим измерения: `dim_company`, `dim_industry`, `dim_region`.
6. Строим факт `fact_financials` с выручкой, прибылью, активами и рентабельностью.
7. Экспортируем в CSV для DataLens.

In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных

Parquet-файлы разбиты по папкам `year=YYYY`. В RFSD около 223 колонок, и если читать файл без фильтра, Arrow развернёт все колонки в память (~7.5 ГБ на файл) и упадёт с `ArrowMemoryError`. Поэтому:
- читаем **только нужные колонки** через `columns=`;
- сразу отсеиваем строки без выручки и с флагом `outlier=True`, чтобы не тащить их в память;
- год берём из имени папки, а не из колонки (в parquet её может не быть).

In [2]:
files = sorted(RAW_DIR.glob("year=*/*.parquet"))
print(f"Найдено файлов: {len(files)}")

years_available = sorted({int(f.parent.name.split("=")[1]) for f in files})
print("Доступные годы:", years_available)

# Читаем только нужные колонки — иначе Arrow разворачивает все 223 и падает по памяти
COLS = ["inn", "okved", "region", "line_2110", "line_2400", "line_1600", "outlier"]

frames = []
for f in files:
    year = int(f.parent.name.split("=")[1])
    print(f"Читаем {f.name} (год {year})...")

    tmp = pd.read_parquet(f, columns=COLS)
    tmp["year"] = year

    # Сразу фильтруем, чтобы не тащить мусор в память
    tmp = tmp[tmp["line_2110"].notna() & (tmp["line_2110"] > 0)]
    tmp = tmp[tmp["outlier"] != True]

    print(f"  осталось строк: {len(tmp):,}")
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
print(f"\nВсего строк: {len(df):,}")

Найдено файлов: 11
Доступные годы: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Читаем part-0.parquet (год 2015)...
  осталось строк: 1,607,306
Читаем part-0.parquet (год 2016)...
  осталось строк: 1,622,351
Читаем part-0.parquet (год 2017)...
  осталось строк: 1,615,390
Читаем part-0.parquet (год 2018)...
  осталось строк: 1,642,366
Читаем part-0.parquet (год 2019)...
  осталось строк: 1,620,204
Читаем part-0.parquet (год 2020)...
  осталось строк: 1,595,613
Читаем part-0.parquet (год 2021)...
  осталось строк: 1,570,207
Читаем part-0.parquet (год 2022)...
  осталось строк: 1,545,897
Читаем part-0.parquet (год 2023)...
  осталось строк: 1,544,907
Читаем part-0.parquet (год 2024)...
  осталось строк: 1,536,119
Читаем part-0.parquet (год 2025)...
  осталось строк: 1,459,558

Всего строк: 17,359,918


## 2. Первичный осмотр

Смотрим, какие колонки есть в датасете, какие типы, где пропуски. Нас интересуют:
- `inn` — идентификатор компании;
- `okved` — код отрасли;
- `region` — код региона;
- `line_2110` — выручка;
- `line_2400` — чистая прибыль;
- `line_1600` — активы (баланс);
- `outlier` — флаг выброса от авторов RFSD.

In [3]:
print("Колонки:", df.columns.tolist())
print("\nТипы:")
print(df.dtypes)

print("\nПропуски по ключевым колонкам (%):")
key_cols = ["inn", "okved", "region", "line_2110", "line_2400", "line_1600", "outlier"]
print((df[key_cols].isnull().mean() * 100).round(2))

Колонки: ['inn', 'okved', 'region', 'line_2110', 'line_2400', 'line_1600', 'outlier', 'year']

Типы:
inn              str
okved            str
region           str
line_2110    float64
line_2400    float64
line_1600    float64
outlier      float64
year           int64
dtype: object

Пропуски по ключевым колонкам (%):
inn          0.00
okved        0.06
region       0.00
line_2110    0.00
line_2400    0.03
line_1600    0.21
outlier      0.00
dtype: float64


## 3. Переименование, приведение типов и перевод в рубли

Технические `line_*` соответствуют строкам бухгалтерского баланса и отчёта о прибылях и убытках. Переименовываем в понятные имена и приводим к числам.

**Единицы измерения.** RFSD хранит финансовые показатели в **тысячах рублей** (стандарт российской бухгалтерской отчётности: строка 2110 «Выручка», строка 2400 «Чистая прибыль», строка 1600 «Активы»). Чтобы дальше работать с привычными «млрд ₽» и «трлн ₽», переводим суммы в рубли умножением на 1000. Показатели-отношения (`profitability`, `roa`) не изменятся.

In [4]:
df = df.rename(columns={
    "line_2110": "revenue",
    "line_2400": "net_profit",
    "line_1600": "assets",
})

# RFSD в тысячах рублей → переводим в рубли
for col in ["revenue", "net_profit", "assets"]:
    df[col] = pd.to_numeric(df[col], errors="coerce") * 1000

print(df[["revenue", "net_profit", "assets"]].describe())

            revenue    net_profit        assets
count  1.735992e+07  1.735469e+07  1.732399e+07
mean   1.613704e+08  8.256205e+06  1.996633e+08
std    1.357844e+10  4.383106e+09  2.139474e+10
min    2.000000e+00 -1.402225e+13 -9.999792e+12
25%    1.926000e+06  3.000000e+03  9.890000e+05
50%    9.009000e+06  1.810000e+05  5.495000e+06
75%    3.768600e+07  1.404000e+06  2.624300e+07
max    3.914984e+13  2.686968e+12  3.306800e+13


## 4. Финальная проверка очистки

Фильтры по выручке и выбросам уже применены в шаге 1. Здесь остаётся проверить, что в данных нет дубликатов `(inn, year)`: одна компания — один год — одна запись. Если число строк до и после совпадает, дубликатов нет.

In [5]:
print(f"До проверки: {len(df):,}")

rows_before = len(df)
df = df.drop_duplicates(subset=["inn", "year"])
dups_removed = rows_before - len(df)

print(f"После проверки: {len(df):,}")
print(f"Удалено дубликатов: {dups_removed}")

До проверки: 17,359,918
После проверки: 17,359,918
Удалено дубликатов: 0


## 5. Проверка покрытия по годам

Важно убедиться, что во всех годах есть сопоставимое число компаний. Если в каком-то году резко меньше записей, это может быть связано с ликвидацией компаний, задержками сдачи отчётности или особенностями сбора RFSD.

In [6]:
year_counts = df.groupby("year")["inn"].nunique().sort_index()
print("Уникальных компаний по годам:")
print(year_counts)

print("\nВсего компаний в выборке:", df["inn"].nunique())

Уникальных компаний по годам:
year
2015    1607306
2016    1622351
2017    1615390
2018    1642366
2019    1620204
2020    1595613
2021    1570207
2022    1545897
2023    1544907
2024    1536119
2025    1459558
Name: inn, dtype: int64

Всего компаний в выборке: 3511156


## 6. Измерения («звезда»)

- **`dim_company`** — уникальные компании с отраслью и регионом.
- **`dim_industry`** — уникальные коды отраслей.
- **`dim_region`** — уникальные коды регионов.

Колонку `company_name` намеренно не создаём: она полностью избыточна (`"Компания " + id`) и раздувает CSV на ~40 МБ. В DataLens при необходимости подпись соберётся вычисляемым полем `"Компания " + [company_id]`.

RFSD анонимизирован: реальных названий компаний нет, но ИНН сохраняется. Коды отраслей и регионов — стандартные ОКВЭД и ОКТМО, их при желании можно расшифровать вручную в DataLens.

In [7]:
# Компании
dim_company = (
    df[["inn", "okved", "region"]]
    .drop_duplicates(subset=["inn"])
    .rename(columns={"inn": "company_id", "okved": "industry_code", "region": "region_code"})
)

# Отрасли
dim_industry = (
    dim_company[["industry_code"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_industry["industry_name"] = "Отрасль " + dim_industry["industry_code"].astype(str)

# Регионы
dim_region = (
    dim_company[["region_code"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_region["region_name"] = "Регион " + dim_region["region_code"].astype(str)

print(f"Компаний: {len(dim_company):,}")
print(f"Отраслей: {len(dim_industry):,}")
print(f"Регионов: {len(dim_region):,}")

Компаний: 3,511,156
Отраслей: 3,210
Регионов: 90


## 7. Факт «финансовые показатели»

В факт кладём ключевые показатели (в рублях) и сразу считаем производные:
- **`profitability`** = чистая прибыль / выручка — рентабельность по чистой прибыли;
- **`roa`** = чистая прибыль / активы — рентабельность активов.

In [8]:
fact = df[["inn", "year", "revenue", "net_profit", "assets"]].copy()
fact = fact.rename(columns={"inn": "company_id"})

# Производные метрики (с защитой от деления на ноль)
fact["profitability"] = fact["net_profit"] / fact["revenue"].replace(0, pd.NA)
fact["roa"] = fact["net_profit"] / fact["assets"].replace(0, pd.NA)

print(fact.head())
print(f"\nСтрок в факте: {len(fact):,}")

   company_id  year     revenue  net_profit      assets  profitability  \
0  4217064462  2015   2204000.0     10000.0   1144000.0       0.004537   
1  4217064543  2015   4026000.0  -5770000.0   4307000.0      -1.433184   
2  4217064670  2015     31000.0         0.0         NaN       0.000000   
3  4217064695  2015     78000.0    -12000.0    140000.0      -0.153846   
4  4217064720  2015  22822000.0   1874000.0  16223000.0       0.082114   

        roa  
0  0.008741  
1  -1.33968  
2       NaN  
3 -0.085714  
4  0.115515  

Строк в факте: 17,359,918


## 8. Экспорт в CSV

Сохраняем четыре CSV в `data/processed/`.

**Важно про размер.** `fact_financials.csv` весит около 1.2 ГБ — он не поместится в Yandex DataLens через подключение «Файлы» (лимит 200 МБ на файл). Для дашборда будем использовать агрегат `fact_yearly_agg.csv` (≈54 МБ): его достаточно для трендов, структуры и карты. Детальный факт нужен только для топ-10 компаний; для этого его можно либо загрузить в PostgreSQL, либо сделать отдельный маленький CSV с топ-100.

In [9]:
dim_company.to_csv(PROCESSED_DIR / "dim_company.csv", index=False)
dim_industry.to_csv(PROCESSED_DIR / "dim_industry.csv", index=False)
dim_region.to_csv(PROCESSED_DIR / "dim_region.csv", index=False)
fact.to_csv(PROCESSED_DIR / "fact_financials.csv", index=False)

print("Файлы сохранены:")
for f in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"  {f.name}: {f.stat().st_size / 1024 / 1024:.1f} МБ")

Файлы сохранены:
  dim_company.csv: 98.7 МБ
  dim_industry.csv: 0.1 МБ
  dim_region.csv: 0.0 МБ
  fact_financials.csv: 1417.7 МБ


## 9. Агрегат для DataLens

Детальный факт слишком большой, в DataLens можно загрузить агрегат по году × отрасли × региону. Этого достаточно для трендов, структуры и карты.

In [10]:
fact_agg = (
    fact
    .merge(dim_company, on="company_id", how="left")
    .groupby(["year", "industry_code", "region_code"], as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        net_profit=("net_profit", "sum"),
        assets=("assets", "sum"),
        companies=("company_id", "nunique"),
    )
)
fact_agg["profitability"] = fact_agg["net_profit"] / fact_agg["revenue"].replace(0, pd.NA)

fact_agg.to_csv(PROCESSED_DIR / "fact_yearly_agg.csv", index=False)
print(f"Агрегат сохранён: {len(fact_agg):,} строк")
print(f"Размер: {(PROCESSED_DIR / 'fact_yearly_agg.csv').stat().st_size / 1024 / 1024:.1f} МБ")

Агрегат сохранён: 809,473 строк
Размер: 60.6 МБ


## Итог

Готовы пять CSV, из которых в DataLens собирается «звезда»:

| Таблица | Роль | Строк | Размер |
|---|---|---|---|---|
| `dim_company` | Измерение | 3 511 156 | 98.7 МБ |
| `dim_industry` | Измерение | 3 210 | 0.1 МБ |
| `dim_region` | Измерение | 90 | 0.0 МБ |
| `fact_financials` | Факт (детальный) | 17 359 918 | 1.4 ГБ |
| `fact_yearly_agg` | Факт (агрегированный) | 809 473 | 60.6 МБ |

Дальше — EDA в `02_data_exploration.ipynb` и сборка дашборда в DataLens (`fact_yearly_agg` + `dim_industry` + `dim_region`).